# MAFEX Demo Notebook

**Morpheme-Aligned Faithful Explanations for Turkish NLP**

This notebook demonstrates the MAFEX framework for generating interpretable explanations aligned with Turkish morphological structure.

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Morphological Analysis

First, let's see how MAFEX analyzes Turkish morphology.

In [ ]:
from mafex.morphology import MorphemeAnalyzer

analyzer = MorphemeAnalyzer()

# Test words
test_words = [
    "yapamayacakmış",  # reportedly, he will not be able to do
    "gelemedim",       # I could not come
    "gözlükçü",        # optician
    "evlerden",        # from the houses
]

for word in test_words:
    analysis = analyzer.analyze_word(word)
    print(f"\n🔤 {word}")
    print(f"   Morphemes: {' + '.join(analysis.morpheme_surfaces)}")
    for m in analysis.morphemes:
        print(f"      • {m.surface} [{m.pos}]")

## 2. Alignment Matrix Construction

The key innovation is the Alignment Matrix **A** that maps tokens to morphemes.

In [ ]:
from mafex.morphology import AlignmentMatrixBuilder

builder = AlignmentMatrixBuilder(analyzer)

# Example: "Gelemedim" with BPE-like tokens
text = "Gelemedim"
tokens = ["Gel", "##eme", "##dim"]  # Simulated BPE tokenization

A, morphemes = builder.build(text, tokens)

print(f"Text: {text}")
print(f"Tokens: {tokens}")
print(f"Morphemes: {morphemes}")
print(f"\nAlignment Matrix A (K={len(morphemes)} x T={len(tokens)}):")
print(A)

# Verify partition property
print(f"\nColumn sums (should be 1): {A.sum(axis=0)}")

## 3. MAFEX Pipeline Demo

Now let's run the full MAFEX pipeline.

In [ ]:
from mafex.models import DemoModelWrapper
from mafex.projection import MAFEXPipeline

# Load demo model (for CPU testing)
wrapper = DemoModelWrapper()
wrapper.load()

# Create MAFEX pipeline
mafex = MAFEXPipeline(
    model=wrapper.model,
    tokenizer=wrapper.tokenizer,
    lambda_causal=0.7,
    ig_steps=20
)

In [ ]:
# Run explanation
text = "Gelemedim"
result = mafex.explain(text)

print(f"Input: {result.text}")
print(f"Target class: {result.target_class}")
print(f"Model output: {result.model_output:.4f}")
print(f"\nMorphemes: {result.morphemes}")
print(f"\nFinal attributions (S*):")
for m, s in zip(result.morphemes, result.final_attributions):
    print(f"  {m}: {s:.4f}")

## 4. Visualization

In [ ]:
from mafex.visualization import create_comparison_plot, create_attribution_heatmap

# Visualize MAFEX result
fig = create_attribution_heatmap(
    result.morphemes,
    result.final_attributions,
    title=f"MAFEX Attribution: '{result.text}'"
)
plt.show()

In [ ]:
# Compare with baseline
fig = create_comparison_plot(
    text=result.text,
    mafex_morphemes=result.morphemes,
    mafex_attributions=result.final_attributions,
    baseline_tokens=result.tokens,
    baseline_attributions=result.token_attributions
)
plt.show()

## 5. Paper Figures

In [ ]:
from mafex.visualization import create_paper_figure_1, create_benchmark_figure

# Figure 1: Fidelity Bottleneck
fig1 = create_paper_figure_1()
plt.show()

In [ ]:
# Figure 3: Benchmark Results
benchmark_results = {
    "BERTurk": {"token_ig": 0.42, "random": 0.50, "mafex": 0.68},
    "Cosmos": {"token_ig": 0.39, "random": 0.47, "mafex": 0.65},
    "Kumru": {"token_ig": 0.45, "random": 0.53, "mafex": 0.71},
    "Aya-23": {"token_ig": 0.41, "random": 0.49, "mafex": 0.69},
}

fig2 = create_benchmark_figure(benchmark_results)
plt.show()

## 6. Batch Processing

In [ ]:
# Process multiple sentences
sentences = [
    "Gelemedim",
    "Yapamayacağız", 
    "Çok güzeldi",
    "Anlamıyorum"
]

results = mafex.explain_batch(sentences, show_progress=True)

for r in results:
    print(f"\n📝 {r.text}")
    print(f"   Top morpheme: {r.get_top_morphemes(1)[0]}")

---

## Summary

MAFEX provides morpheme-aligned explanations that are more faithful and interpretable than token-level baselines for Turkish and other morphologically rich languages.

**Key components:**
1. Morphological Analyzer (Zemberek-based)
2. Alignment Matrix A (token→morpheme mapping)
3. Projection: φ_morph = A · φ_tok
4. Causal Regularization: S* = λ·φ_morph + (1-λ)·φ_causal